# Evaluate held-out events and inspect results

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from IPython.display import display, Image

root = Path.cwd().resolve()
if root.name == "notebooks":
    root = root.parent
if not (root / "src/data.py").exists():
    raise RuntimeError("Launch Jupyter from the repository or notebooks folder.")
sys.path.insert(0, str(root))
from src.data import ROOT, LEVELS, load, mix

Apply identical inputs to every method. This notebook uses the refitted head from notebook 06 if available, otherwise it uses the bundled head.

In [2]:
from src.evaluate import evaluate

head = ROOT / "runs/notebook_fit/das_head.npz"
if not head.exists():
    head = ROOT / "models/das_head.npz"
out = ROOT / "runs/notebook_test"
evaluate(ROOT / "data/prepared", out, head, save_predictions=True)

                  gain_db     cc   rmse  nrmse  ms_per_trace  events  onset_mae_ms  pick_recovery  reference_picks  matched_picks
level method                                                                                                                     
-5    Bandpass      0.204  0.340  0.136  1.750         0.039     145        76.208          0.925             3466           3207
      Fine-tuned    0.373  0.489  0.139  1.704         7.578     145        71.927          0.927             3466           3213
      Noisy         0.000  0.485  0.148  1.778         0.000     145        74.525          0.923             3466           3198
      Scalar        5.155  0.485  0.082  0.983         0.000     145        74.525          0.923             3466           3198
      Wavelet       0.242  0.479  0.144  1.730         0.015     145        73.622          0.930             3466           3222
      Zero-shot     0.069  0.487  0.146  1.764         7.590     145        74.078        

In [3]:
df = pd.read_csv(out / "summary.csv")
display(df.pivot(index="method", columns="level", values="gain_db").round(2))
paired = pd.read_csv(out / "paired_gain.csv")
display(paired[paired.baseline == "Zero-shot"].round(3))
sites = pd.read_csv(out / "by_array.csv")
display(sites[sites.level == 0].pivot(index="method", columns="array", values="gain_db").round(2))

level,-5,0,5,10
method,,,,
Bandpass,0.20,-1.05,-3.55,-7.29
Fine-tuned,0.37,0.40,0.42,0.41
Noisy,0.00,0.00,0.00,0.00
Scalar,5.15,3.04,-0.44,-4.83
Wavelet,0.24,0.20,0.13,-0.01
Zero-shot,0.07,0.08,0.09,0.11
f-k,3.16,1.74,-0.86,-4.58


,level,baseline,events,tuned_minus_baseline_db,ci95_low_db,ci95_high_db,events_tuned_wins
0,-5,Zero-shot,145,0.304,0.284,0.327,145
3,0,Zero-shot,145,0.319,0.296,0.345,145
6,5,Zero-shot,145,0.332,0.309,0.359,145
9,10,Zero-shot,145,0.305,0.278,0.334,145


array,longvalley,ridgecrest
method,,
Bandpass,-1.08,-0.69
Fine-tuned,0.39,0.42
Noisy,0.00,0.00
Scalar,3.05,2.93
Wavelet,0.21,0.14
Zero-shot,0.08,0.09
f-k,1.76,1.44


Inspect both the average gain and between-event variation. The committed report includes timing error, recovery fraction, RMSE and waveform correlation.